In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistData") \
    .getOrCreate()
spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/19 14:05:45 INFO SparkEnv: Registering MapOutputTracker
26/07/19 14:05:45 INFO SparkEnv: Registering BlockManagerMaster
26/07/19 14:05:45 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/07/19 14:05:45 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
hdfs_path = "/data/olist/"

In [3]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema=True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv', header=True, inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv', header=True, inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv', header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv', header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv', header=True, inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv', header=True, inferSchema=True)

In [4]:
# Caching frequently used data for better performance

orders_df.cache()
customers_df.cache()
order_item_df.cache()

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [5]:
order_item_joined_df = orders_df.join(order_item_df,'order_id','inner')

In [7]:
order_item_products_df = order_item_joined_df.join(products_df,'product_id','inner')

In [8]:
order_item_products_seller_df = order_item_products_df.join(sellers_df,'seller_id','inner')

In [9]:
full_orders_df = order_item_products_seller_df.join(customers_df,'customer_id','inner')

In [10]:
#GeoLocation Data

full_orders_df = full_orders_df.join(geolocation_df,full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [11]:
full_orders_df = full_orders_df.join(reviews_df,'order_id','left')

In [12]:
full_orders_df = full_orders_df.join(payments_df,'order_id','left')

In [13]:
full_orders_df.cache()

26/07/19 14:23:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [19]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [16]:
from pyspark.sql.functions import *

In [17]:
#TOTAL REVENUES PER SELLER 

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price'))

In [18]:
seller_revenue_df.show(5)

+--------------------+--------------------+
|           seller_id|          sum(price)|
+--------------------+--------------------+
|3d5d0dc7073a299e3...|  170639.59999999998|
|2138ccb85b11a4ec1...|   1943866.069999998|
|33ac3e28642ab8bda...|   615628.8499999985|
|cc419e0650a3c5ba7...|1.4751464500000045E7|
|b76dba6c951ab00dc...|   302582.6599999992|
+--------------------+--------------------+
only showing top 5 rows

